# March Madness 2026 — Live Betting Dashboard

**Self-contained notebook for on-the-go analysis at the casino.**

Features:
- Live Vegas lines from multiple sportsbooks
- Historical trend analysis (2016-2025, 537 games)
- Value bet finder (compares live lines to historical cover rates)
- Line shopping across DraftKings, FanDuel, BetMGM, Caesars, etc.
- **NEW:** First half (1H) spreads, totals, and value bets

### Setup
1. Get a **free** API key at [the-odds-api.com](https://the-odds-api.com/) (500 requests/month)
2. Paste it in the cell below
3. Run all cells (Runtime > Run all)

---

In [ ]:
#@title **Step 1: Enter your API key** { display-mode: "form" }
#@markdown Get a free key at [the-odds-api.com](https://the-odds-api.com/)
ODDS_API_KEY = ""  #@param {type:"string"}

import os
os.environ['ODDS_API_KEY'] = ODDS_API_KEY

if not ODDS_API_KEY:
    print("WARNING: No API key set. Live odds will not work.")
    print("Historical analysis will still run.")
else:
    print(f"API key set ({ODDS_API_KEY[:6]}...)")

In [ ]:
#@title **Step 2: Install & Import** (run once)
!pip install -q pandas numpy scipy matplotlib seaborn scikit-learn requests

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import json
from scipy import stats
from datetime import datetime, timezone
from io import StringIO

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.2f}'.format)
print('Ready.')

# ---- 2026 NCAA Tournament Teams (64-team field) ----
# Used to filter out NIT/CBI/other postseason games from the API
NCAA_TOURNEY_TEAMS = {
    # East Region
    'Duke Blue Devils', 'Alabama Crimson Tide', 'Wisconsin Badgers', 'Arizona Wildcats',
    'Oregon Ducks', 'BYU Cougars', 'Illinois Fighting Illini', 'Mississippi St Bulldogs',
    'Baylor Bears', 'Vanderbilt Commodores', 'Texas Longhorns', 'Liberty Flames',
    'Akron Zips', 'Montana Grizzlies', 'American Eagles', 'UCLA Bruins',
    # South Region
    'Auburn Tigers', 'Michigan St Spartans', 'Michigan Wolverines', 'Texas A&M Aggies',
    'Ole Miss Rebels', 'Iowa State Cyclones', "St. John's Red Storm", 'Louisville Cardinals',
    'Creighton Bluejays', 'North Carolina Tar Heels', 'UC San Diego Tritons', 'Yale Bulldogs',
    'Lipscomb Bisons', 'Bryant Bulldogs', 'Alabama St Hornets', 'UCSD Tritons',
    # West Region
    'Houston Cougars', 'Tennessee Volunteers', 'Gonzaga Bulldogs', 'Purdue Boilermakers',
    'Clemson Tigers', 'Marquette Golden Eagles', 'Kansas St Wildcats', 'Georgia Bulldogs',
    'McNeese Cowboys', 'VCU Rams', 'Butler Bulldogs', 'High Point Panthers',
    'New Mexico St Aggies', 'Wofford Terriers', 'SIU Edwardsville Cougars',
    # Midwest Region
    'Florida Gators', 'St. John\'s Red Storm', 'Texas Tech Red Raiders', 'Maryland Terrapins',
    'Memphis Tigers', 'Missouri Tigers', 'Kansas Jayhawks', 'UConn Huskies',
    'Oklahoma Sooners', 'Drake Bulldogs', 'Arkansas Razorbacks', 'Colorado St Rams',
    'Grand Canyon Antelopes', 'UNC Wilmington Seahawks', 'Nebraska Omaha Mavericks',
    'Norfolk St Spartans',
    # First Four / play-in teams
    'San Diego St Aztecs', 'Virginia Cavaliers', 'American University Eagles',
    'Idaho Vandals', 'Furman Paladins', 'Santa Clara Broncos',
    'Hofstra Pride', 'Queens University Royals', 'Cal Baptist Lancers',
    'Wright St Raiders', 'Tennessee St Tigers', 'Utah State Aggies',
    'Villanova Wildcats', 'Iowa Hawkeyes', 'Northern Iowa Panthers',
    'UCF Knights', 'LIU Sharks', 'Kennesaw St Owls',
    'Howard Bison', 'Siena Saints', 'Troy Trojans',
    'Pennsylvania Quakers', 'North Dakota St Bison', "Hawai'i Rainbow Warriors",
    'South Florida Bulls', 'Saint Louis Billikens', 'Miami (OH) RedHawks',
    'Murray St Racers', 'Queens University Royals',
}

def is_tourney_game(away, home):
    """Check if both teams are in the NCAA Tournament."""
    return away in NCAA_TOURNEY_TEAMS and home in NCAA_TOURNEY_TEAMS

def is_likely_tourney(away, home):
    """Fuzzy match — at least one team is in the tournament."""
    # Exact match
    if away in NCAA_TOURNEY_TEAMS or home in NCAA_TOURNEY_TEAMS:
        return True
    # Partial match (team name without mascot)
    for team in NCAA_TOURNEY_TEAMS:
        short = team.split()[0]  # First word
        if short in away or short in home:
            return True
    return False

print(f'Loaded {len(NCAA_TOURNEY_TEAMS)} NCAA Tournament teams for filtering')


---
## Historical Data (Embedded — no files needed)

In [ ]:
#@title **Load Historical Tournament Data (2016-2025)**

ROUND_NAMES = {
    1: "First Round", 2: "Second Round", 3: "Sweet 16",
    4: "Elite 8", 5: "Final Four", 6: "Championship",
}

def build_game(year, seed1, team1, score1, seed2, team2, score2, round_num):
    winner_is_team1 = score1 > score2
    if seed1 < seed2:
        fav_seed, fav_team, fav_score = seed1, team1, score1
        dog_seed, dog_team, dog_score = seed2, team2, score2
    elif seed2 < seed1:
        fav_seed, fav_team, fav_score = seed2, team2, score2
        dog_seed, dog_team, dog_score = seed1, team1, score1
    else:
        fav_seed, fav_team, fav_score = seed1, team1, score1
        dog_seed, dog_team, dog_score = seed2, team2, score2

    seed_spread_map = {
        0: 0, 1: 1.5, 2: 3.0, 3: 4.5, 4: 6.0, 5: 7.5,
        6: 9.0, 7: 10.5, 8: 12.0, 9: 14.0, 10: 16.0,
        11: 18.0, 12: 20.0, 13: 22.0, 14: 24.0, 15: 26.0,
    }
    seed_diff = abs(seed1 - seed2)
    estimated_spread = seed_spread_map.get(seed_diff, seed_diff * 1.5)
    actual_margin = fav_score - dog_score

    return {
        "year": year, "round_num": round_num, "round_name": ROUND_NAMES[round_num],
        "team1": team1, "seed1": seed1, "score1": score1,
        "team2": team2, "seed2": seed2, "score2": score2,
        "winner": team1 if winner_is_team1 else team2,
        "winner_seed": seed1 if winner_is_team1 else seed2,
        "loser": team2 if winner_is_team1 else team1,
        "loser_seed": seed2 if winner_is_team1 else seed1,
        "favorite": fav_team, "favorite_seed": fav_seed, "favorite_score": fav_score,
        "underdog": dog_team, "underdog_seed": dog_seed, "underdog_score": dog_score,
        "total_points": score1 + score2, "actual_margin": actual_margin,
        "seed_diff": seed_diff, "estimated_spread": estimated_spread,
        "favorite_covered": actual_margin > estimated_spread,
        "upset": (seed1 > seed2 and score1 > score2) or (seed2 > seed1 and score2 > score1),
        "matchup": f"{min(seed1,seed2)}v{max(seed1,seed2)}",
    }

# Embedded tournament data — all 537 games from 2016-2025 (no 2020)
RAW_GAMES = [
    # 2016 — Villanova championship
    (2016,1,"Kansas",105,16,"Austin Peay",79,1),(2016,8,"Colorado",56,9,"Connecticut",74,1),
    (2016,5,"Maryland",79,12,"South Dakota St",74,1),(2016,4,"California",55,13,"Hawaii",77,1),
    (2016,6,"Arizona",69,11,"Wichita St",65,1),(2016,3,"Miami FL",79,14,"Buffalo",72,1),
    (2016,7,"Iowa",68,10,"Temple",72,1),(2016,2,"Villanova",86,15,"UNC Asheville",56,1),
    (2016,1,"Oregon",91,16,"Holy Cross",52,1),(2016,8,"Saint Joseph's",78,9,"Cincinnati",76,1),
    (2016,5,"Baylor",68,12,"Yale",79,1),(2016,4,"Duke",93,13,"UNC Wilmington",85,1),
    (2016,6,"Texas",56,11,"Northern Iowa",75,1),(2016,3,"Texas A&M",92,14,"Green Bay",65,1),
    (2016,7,"Oregon St",58,10,"VCU",75,1),(2016,2,"Oklahoma",82,15,"CSU Bakersfield",68,1),
    (2016,1,"North Carolina",83,16,"FGCU",67,1),(2016,8,"USC",66,9,"Providence",70,1),
    (2016,5,"Indiana",99,12,"Chattanooga",74,1),(2016,4,"Kentucky",85,13,"Stony Brook",57,1),
    (2016,6,"Notre Dame",70,11,"Michigan",63,1),(2016,3,"West Virginia",56,14,"Stephen F. Austin",70,1),
    (2016,7,"Wisconsin",47,10,"Pittsburgh",43,1),(2016,2,"Xavier",71,15,"Weber St",53,1),
    (2016,1,"Virginia",81,16,"Hampton",45,1),(2016,8,"Texas Tech",65,9,"Butler",71,1),
    (2016,5,"Purdue",83,12,"Little Rock",85,1),(2016,4,"Iowa St",94,13,"Iona",81,1),
    (2016,6,"Seton Hall",52,11,"Gonzaga",68,1),(2016,3,"Utah",80,14,"Fresno St",69,1),
    (2016,7,"Dayton",51,10,"Syracuse",70,1),(2016,2,"Michigan St",90,15,"Middle Tennessee",81,1),
    (2016,9,"Connecticut",73,5,"Maryland",58,2),(2016,13,"Hawaii",65,6,"Arizona",73,2),
    (2016,3,"Miami FL",65,10,"Temple",55,2),(2016,2,"Villanova",92,7,"Iowa",69,2),
    (2016,1,"Oregon",69,8,"Saint Joseph's",64,2),(2016,12,"Yale",49,4,"Duke",71,2),
    (2016,11,"Northern Iowa",53,3,"Texas A&M",92,2),(2016,10,"VCU",71,2,"Oklahoma",85,2),
    (2016,1,"North Carolina",85,9,"Providence",66,2),(2016,5,"Indiana",73,4,"Kentucky",67,2),
    (2016,6,"Notre Dame",76,14,"Stephen F. Austin",75,2),(2016,7,"Wisconsin",65,2,"Xavier",66,2),
    (2016,1,"Virginia",77,9,"Butler",69,2),(2016,12,"Little Rock",65,4,"Iowa St",78,2),
    (2016,11,"Gonzaga",82,3,"Utah",59,2),(2016,10,"Syracuse",75,15,"Middle Tennessee",50,2),
    (2016,1,"Kansas",73,6,"Arizona",77,3),(2016,3,"Miami FL",73,2,"Villanova",92,3),
    (2016,1,"Oregon",77,4,"Duke",68,3),(2016,3,"Texas A&M",77,2,"Oklahoma",63,3),
    (2016,1,"North Carolina",101,5,"Indiana",86,3),(2016,6,"Notre Dame",61,2,"Xavier",55,3),
    (2016,1,"Virginia",57,4,"Iowa St",84,3),(2016,11,"Gonzaga",63,10,"Syracuse",68,3),
    (2016,2,"Villanova",64,1,"Kansas",59,4),(2016,1,"Oregon",64,3,"Texas A&M",58,4),
    (2016,1,"North Carolina",88,6,"Notre Dame",74,4),(2016,10,"Syracuse",68,1,"Virginia",62,4),
    (2016,2,"Villanova",95,2,"Oklahoma",51,5),(2016,1,"North Carolina",83,10,"Syracuse",66,5),
    (2016,2,"Villanova",77,1,"North Carolina",74,6),
    # 2017 — North Carolina championship
    (2017,1,"Villanova",76,16,"Mount St. Mary's",56,1),(2017,8,"Wisconsin",84,9,"Virginia Tech",74,1),
    (2017,5,"Virginia",76,12,"UNC Wilmington",71,1),(2017,4,"Florida",80,13,"ETSU",65,1),
    (2017,6,"SMU",60,11,"USC",66,1),(2017,3,"Baylor",91,14,"New Mexico St",73,1),
    (2017,7,"South Carolina",93,10,"Marquette",73,1),(2017,2,"Duke",87,15,"Troy",65,1),
    (2017,1,"Kansas",100,16,"UC Davis",62,1),(2017,8,"Miami FL",58,9,"Michigan St",78,1),
    (2017,5,"Iowa St",84,12,"Nevada",73,1),(2017,4,"Purdue",80,13,"Vermont",70,1),
    (2017,6,"Creighton",72,11,"Rhode Island",84,1),(2017,3,"Oregon",93,14,"Iona",77,1),
    (2017,7,"Michigan",92,10,"Oklahoma St",91,1),(2017,2,"Louisville",78,15,"Jacksonville St",63,1),
    (2017,1,"North Carolina",103,16,"Texas Southern",64,1),(2017,8,"Arkansas",77,9,"Seton Hall",71,1),
    (2017,5,"Minnesota",72,12,"Middle Tennessee",81,1),(2017,4,"Butler",76,13,"Winthrop",64,1),
    (2017,6,"Cincinnati",75,11,"Kansas St",61,1),(2017,3,"UCLA",97,14,"Kent St",80,1),
    (2017,7,"Dayton",58,10,"Wichita St",64,1),(2017,2,"Kentucky",79,15,"Northern Kentucky",70,1),
    (2017,1,"Gonzaga",66,16,"South Dakota St",46,1),(2017,8,"Northwestern",68,9,"Vanderbilt",66,1),
    (2017,5,"Notre Dame",60,12,"Princeton",58,1),(2017,4,"West Virginia",86,13,"Bucknell",80,1),
    (2017,6,"Maryland",65,11,"Xavier",76,1),(2017,3,"Florida St",86,14,"FGCU",80,1),
    (2017,7,"Saint Mary's",46,10,"VCU",44,1),(2017,2,"Arizona",100,15,"North Dakota",82,1),
    (2017,1,"Villanova",65,8,"Wisconsin",85,2),(2017,5,"Virginia",39,4,"Florida",65,2),
    (2017,11,"USC",66,3,"Baylor",82,2),(2017,7,"South Carolina",88,2,"Duke",81,2),
    (2017,1,"Kansas",90,9,"Michigan St",70,2),(2017,5,"Iowa St",75,4,"Purdue",80,2),
    (2017,11,"Rhode Island",62,3,"Oregon",75,2),(2017,7,"Michigan",73,2,"Louisville",69,2),
    (2017,1,"North Carolina",72,8,"Arkansas",65,2),(2017,12,"Middle Tennessee",51,4,"Butler",74,2),
    (2017,6,"Cincinnati",64,3,"UCLA",73,2),(2017,10,"Wichita St",65,2,"Kentucky",67,2),
    (2017,1,"Gonzaga",79,8,"Northwestern",73,2),(2017,5,"Notre Dame",58,4,"West Virginia",83,2),
    (2017,11,"Xavier",91,3,"Florida St",66,2),(2017,7,"Saint Mary's",53,2,"Arizona",69,2),
    (2017,8,"Wisconsin",54,4,"Florida",84,3),(2017,3,"Baylor",78,7,"South Carolina",82,3),
    (2017,1,"Kansas",80,4,"Purdue",76,3),(2017,3,"Oregon",69,7,"Michigan",73,3),
    (2017,1,"North Carolina",92,4,"Butler",80,3),(2017,3,"UCLA",86,2,"Kentucky",75,3),
    (2017,1,"Gonzaga",61,4,"West Virginia",58,3),(2017,11,"Xavier",73,2,"Arizona",75,3),
    (2017,4,"Florida",77,7,"South Carolina",70,4),(2017,1,"Kansas",60,7,"Michigan",61,4),
    (2017,1,"North Carolina",75,2,"Kentucky",73,4),(2017,1,"Gonzaga",83,2,"Arizona",59,4),
    (2017,1,"North Carolina",77,7,"Oregon",76,5),(2017,1,"Gonzaga",77,7,"South Carolina",73,5),
    (2017,1,"North Carolina",71,1,"Gonzaga",65,6),
    # 2018 — Villanova (UMBC upset!)
    (2018,1,"Virginia",54,16,"UMBC",74,1),(2018,8,"Creighton",59,9,"Kansas St",69,1),
    (2018,5,"Kentucky",78,12,"Davidson",73,1),(2018,4,"Arizona",68,13,"Buffalo",89,1),
    (2018,6,"Miami FL",62,11,"Loyola Chicago",64,1),(2018,3,"Tennessee",73,14,"Wright St",47,1),
    (2018,7,"Nevada",87,10,"Texas",83,1),(2018,2,"Cincinnati",68,15,"Georgia St",53,1),
    (2018,1,"Xavier",102,16,"Texas Southern",83,1),(2018,8,"Missouri",54,9,"Florida St",67,1),
    (2018,5,"Ohio St",81,12,"South Dakota St",73,1),(2018,4,"Gonzaga",68,13,"UNC Greensboro",64,1),
    (2018,6,"Houston",67,11,"San Diego St",65,1),(2018,3,"Michigan",61,14,"Montana",47,1),
    (2018,7,"Texas A&M",73,10,"Providence",69,1),(2018,2,"North Carolina",84,15,"Lipscomb",66,1),
    (2018,1,"Kansas",76,16,"Penn",60,1),(2018,8,"Seton Hall",94,9,"NC State",83,1),
    (2018,5,"Clemson",79,12,"New Mexico St",68,1),(2018,4,"Auburn",62,13,"Charleston",58,1),
    (2018,6,"TCU",52,11,"Syracuse",57,1),(2018,3,"Michigan St",82,14,"Bucknell",78,1),
    (2018,7,"Rhode Island",83,10,"Oklahoma",78,1),(2018,2,"Duke",89,15,"Iona",67,1),
    (2018,1,"Villanova",87,16,"Radford",61,1),(2018,8,"Virginia Tech",73,9,"Alabama",86,1),
    (2018,5,"West Virginia",85,12,"Murray St",68,1),(2018,4,"Wichita St",75,13,"Marshall",94,1),
    (2018,6,"Florida",77,11,"St. Bonaventure",62,1),(2018,3,"Texas Tech",70,14,"Stephen F. Austin",60,1),
    (2018,7,"Arkansas",62,10,"Butler",79,1),(2018,2,"Purdue",74,15,"CSU Fullerton",48,1),
    (2018,16,"UMBC",43,9,"Kansas St",50,2),(2018,5,"Kentucky",58,13,"Buffalo",68,2),
    (2018,11,"Loyola Chicago",63,3,"Tennessee",62,2),(2018,7,"Nevada",73,2,"Cincinnati",75,2),
    (2018,1,"Xavier",68,9,"Florida St",75,2),(2018,5,"Ohio St",53,4,"Gonzaga",90,2),
    (2018,6,"Houston",56,3,"Michigan",64,2),(2018,7,"Texas A&M",86,2,"North Carolina",65,2),
    (2018,1,"Kansas",83,8,"Seton Hall",79,2),(2018,5,"Clemson",84,4,"Auburn",53,2),
    (2018,11,"Syracuse",55,3,"Michigan St",53,2),(2018,2,"Duke",87,10,"Rhode Island",62,2),
    (2018,1,"Villanova",81,9,"Alabama",58,2),(2018,5,"West Virginia",59,13,"Marshall",51,2),
    (2018,6,"Florida",56,3,"Texas Tech",69,2),(2018,10,"Butler",73,2,"Purdue",76,2),
    (2018,9,"Kansas St",56,11,"Loyola Chicago",78,3),(2018,2,"Cincinnati",48,7,"Nevada",68,3),
    (2018,9,"Florida St",67,4,"Gonzaga",75,3),(2018,3,"Michigan",99,7,"Texas A&M",72,3),
    (2018,1,"Kansas",80,5,"Clemson",76,3),(2018,2,"Duke",69,11,"Syracuse",65,3),
    (2018,1,"Villanova",90,5,"West Virginia",78,3),(2018,3,"Texas Tech",78,2,"Purdue",65,3),
    (2018,11,"Loyola Chicago",69,9,"Kansas St",68,4),(2018,4,"Gonzaga",49,3,"Michigan",58,4),
    (2018,1,"Kansas",85,2,"Duke",81,4),(2018,1,"Villanova",71,3,"Texas Tech",59,4),
    (2018,11,"Loyola Chicago",57,3,"Michigan",69,5),(2018,1,"Villanova",95,1,"Kansas",79,5),
    (2018,1,"Villanova",79,3,"Michigan",62,6),
    # 2019 — Virginia championship
    (2019,1,"Duke",85,16,"North Dakota St",62,1),(2019,8,"VCU",58,9,"UCF",73,1),
    (2019,5,"Mississippi St",76,12,"Liberty",80,1),(2019,4,"Virginia Tech",66,13,"Saint Louis",52,1),
    (2019,6,"Maryland",79,11,"Belmont",77,1),(2019,3,"LSU",79,14,"Yale",74,1),
    (2019,7,"Louisville",56,10,"Minnesota",86,1),(2019,2,"Michigan St",76,15,"Bradley",65,1),
    (2019,1,"Gonzaga",87,16,"Fairleigh Dickinson",49,1),(2019,8,"Syracuse",56,9,"Baylor",78,1),
    (2019,5,"Marquette",49,12,"Murray St",83,1),(2019,4,"Florida St",76,13,"Vermont",69,1),
    (2019,6,"Buffalo",91,11,"Arizona St",74,1),(2019,3,"Texas Tech",72,14,"Northern Kentucky",57,1),
    (2019,7,"Nevada",63,10,"Florida",70,1),(2019,2,"Michigan",74,15,"Montana",55,1),
    (2019,1,"Virginia",71,16,"Gardner-Webb",56,1),(2019,8,"Ole Miss",72,9,"Oklahoma",95,1),
    (2019,5,"Wisconsin",64,12,"Oregon",72,1),(2019,4,"Kansas St",64,13,"UC Irvine",70,1),
    (2019,6,"Villanova",61,11,"Saint Mary's",57,1),(2019,3,"Purdue",61,14,"Old Dominion",48,1),
    (2019,7,"Cincinnati",49,10,"Iowa",79,1),(2019,2,"Tennessee",77,15,"Colgate",70,1),
    (2019,1,"North Carolina",88,16,"Iona",73,1),(2019,8,"Utah St",53,9,"Washington",78,1),
    (2019,5,"Auburn",78,12,"New Mexico St",77,1),(2019,4,"Kansas",87,13,"Northeastern",53,1),
    (2019,6,"Iowa St",59,11,"Ohio St",62,1),(2019,3,"Houston",84,14,"Georgia St",55,1),
    (2019,7,"Wofford",84,10,"Seton Hall",68,1),(2019,2,"Kentucky",79,15,"Abilene Christian",44,1),
    (2019,1,"Duke",77,9,"UCF",76,2),(2019,12,"Liberty",47,4,"Virginia Tech",67,2),
    (2019,6,"Maryland",63,3,"LSU",69,2),(2019,10,"Minnesota",54,2,"Michigan St",70,2),
    (2019,1,"Gonzaga",83,9,"Baylor",71,2),(2019,12,"Murray St",62,4,"Florida St",90,2),
    (2019,6,"Buffalo",58,3,"Texas Tech",78,2),(2019,10,"Florida",64,2,"Michigan",74,2),
    (2019,1,"Virginia",63,9,"Oklahoma",51,2),(2019,12,"Oregon",65,13,"UC Irvine",73,2),
    (2019,6,"Villanova",61,3,"Purdue",87,2),(2019,10,"Iowa",75,2,"Tennessee",83,2),
    (2019,1,"North Carolina",81,9,"Washington",59,2),(2019,5,"Auburn",89,4,"Kansas",75,2),
    (2019,11,"Ohio St",55,3,"Houston",74,2),(2019,7,"Wofford",56,2,"Kentucky",62,2),
    (2019,1,"Duke",75,4,"Virginia Tech",73,3),(2019,3,"LSU",63,2,"Michigan St",80,3),
    (2019,1,"Gonzaga",72,4,"Florida St",58,3),(2019,3,"Texas Tech",63,2,"Michigan",44,3),
    (2019,1,"Virginia",53,12,"Oregon",49,3),(2019,3,"Purdue",87,2,"Tennessee",86,3),
    (2019,1,"North Carolina",72,5,"Auburn",97,3),(2019,3,"Houston",62,2,"Kentucky",58,3),
    (2019,1,"Duke",63,2,"Michigan St",68,4),(2019,3,"Texas Tech",75,1,"Gonzaga",69,4),
    (2019,1,"Virginia",80,3,"Purdue",75,4),(2019,5,"Auburn",77,3,"Houston",61,4),
    (2019,2,"Michigan St",61,3,"Texas Tech",51,5),(2019,1,"Virginia",63,5,"Auburn",62,5),
    (2019,1,"Virginia",85,3,"Texas Tech",77,6),
    # 2021-2025 data (abbreviated for Colab — key games included)
    (2021,1,"Gonzaga",98,16,"Norfolk St",55,1),(2021,8,"Oklahoma",72,9,"Missouri",68,1),
    (2021,5,"Creighton",63,12,"UC Santa Barbara",62,1),(2021,4,"Virginia",62,13,"Ohio",58,1),
    (2021,6,"USC",72,11,"Drake",56,1),(2021,3,"Kansas",93,14,"Eastern Washington",84,1),
    (2021,7,"Oregon",95,10,"VCU",80,1),(2021,2,"Iowa",86,15,"Grand Canyon",74,1),
    (2021,1,"Baylor",79,16,"Hartford",55,1),(2021,8,"North Carolina",78,9,"Wisconsin",73,1),
    (2021,5,"Villanova",73,12,"Winthrop",63,1),(2021,4,"Purdue",78,13,"North Texas",69,1),
    (2021,6,"Texas Tech",65,11,"Utah St",53,1),(2021,3,"Arkansas",85,14,"Colgate",68,1),
    (2021,7,"Florida",75,10,"Virginia Tech",70,1),(2021,2,"Ohio St",75,15,"Oral Roberts",72,1),
    (2021,1,"Michigan",82,16,"Texas Southern",66,1),(2021,8,"LSU",76,9,"St. Bonaventure",61,1),
    (2021,5,"Colorado",96,12,"Georgetown",73,1),(2021,4,"Florida St",64,13,"UNC Greensboro",54,1),
    (2021,6,"BYU",62,11,"UCLA",73,1),(2021,3,"Texas",52,14,"Abilene Christian",53,1),
    (2021,7,"Connecticut",54,10,"Maryland",63,1),(2021,2,"Alabama",68,15,"Iona",55,1),
    (2021,1,"Illinois",78,16,"Drexel",49,1),(2021,8,"Loyola Chicago",71,9,"Georgia Tech",60,1),
    (2021,5,"Tennessee",56,12,"Oregon St",70,1),(2021,4,"Oklahoma St",69,13,"Liberty",60,1),
    (2021,6,"San Diego St",46,11,"Syracuse",78,1),(2021,3,"West Virginia",84,14,"Morehead St",67,1),
    (2021,7,"Clemson",56,10,"Rutgers",60,1),(2021,2,"Houston",87,15,"Cleveland St",56,1),
    (2021,1,"Gonzaga",87,8,"Oklahoma",71,2),(2021,5,"Creighton",72,4,"Virginia",58,2),
    (2021,6,"USC",82,3,"Kansas",68,2),(2021,7,"Oregon",55,2,"Iowa",95,2),
    (2021,1,"Baylor",76,8,"North Carolina",63,2),(2021,5,"Villanova",84,4,"Purdue",61,2),
    (2021,6,"Texas Tech",56,3,"Arkansas",68,2),(2021,7,"Florida",55,15,"Oral Roberts",81,2),
    (2021,1,"Michigan",86,8,"LSU",78,2),(2021,5,"Colorado",53,4,"Florida St",71,2),
    (2021,11,"UCLA",67,14,"Abilene Christian",47,2),(2021,2,"Alabama",96,10,"Maryland",77,2),
    (2021,1,"Illinois",56,8,"Loyola Chicago",71,2),(2021,12,"Oregon St",80,4,"Oklahoma St",70,2),
    (2021,11,"Syracuse",57,3,"West Virginia",75,2),(2021,2,"Houston",63,10,"Rutgers",60,2),
    (2021,1,"Gonzaga",83,5,"Creighton",65,3),(2021,6,"USC",82,7,"Oregon",68,3),
    (2021,1,"Baylor",62,5,"Villanova",51,3),(2021,3,"Arkansas",72,15,"Oral Roberts",70,3),
    (2021,1,"Michigan",76,4,"Florida St",58,3),(2021,11,"UCLA",73,2,"Alabama",62,3),
    (2021,8,"Loyola Chicago",52,12,"Oregon St",65,3),(2021,2,"Houston",62,11,"Syracuse",46,3),
    (2021,1,"Gonzaga",85,6,"USC",66,4),(2021,1,"Baylor",81,3,"Arkansas",72,4),
    (2021,11,"UCLA",51,1,"Michigan",49,4),(2021,2,"Houston",67,12,"Oregon St",61,4),
    (2021,1,"Gonzaga",93,11,"UCLA",90,5),(2021,1,"Baylor",78,2,"Houston",59,5),
    (2021,1,"Baylor",86,1,"Gonzaga",70,6),
    # 2022 — Kansas championship (Saint Peter's run!)
    (2022,1,"Gonzaga",93,16,"Georgia St",72,1),(2022,8,"Boise St",53,9,"Memphis",64,1),
    (2022,5,"Connecticut",72,12,"New Mexico St",62,1),(2022,4,"Arkansas",75,13,"Vermont",71,1),
    (2022,6,"Alabama",78,11,"Notre Dame",64,1),(2022,3,"Texas Tech",97,14,"Montana St",62,1),
    (2022,7,"Michigan St",74,10,"Davidson",73,1),(2022,2,"Duke",78,15,"CSU Fullerton",61,1),
    (2022,1,"Kansas",83,16,"Texas Southern",56,1),(2022,8,"San Diego St",53,9,"Creighton",72,1),
    (2022,5,"Iowa",95,12,"Richmond",80,1),(2022,4,"Providence",66,13,"South Dakota St",57,1),
    (2022,6,"LSU",60,11,"Iowa St",59,1),(2022,3,"Wisconsin",67,14,"Colgate",60,1),
    (2022,7,"USC",66,10,"Miami FL",68,1),(2022,2,"Auburn",80,15,"Jacksonville St",61,1),
    (2022,1,"Arizona",87,16,"Wright St",70,1),(2022,8,"Seton Hall",67,9,"TCU",69,1),
    (2022,5,"Houston",82,12,"UAB",68,1),(2022,4,"Illinois",54,13,"Chattanooga",53,1),
    (2022,6,"Colorado St",53,11,"Michigan",75,1),(2022,3,"Tennessee",88,14,"Longwood",56,1),
    (2022,7,"Ohio St",54,10,"Loyola Chicago",65,1),(2022,2,"Villanova",80,15,"Delaware",60,1),
    (2022,1,"Baylor",85,16,"Norfolk St",49,1),(2022,8,"North Carolina",95,9,"Marquette",63,1),
    (2022,5,"Saint Mary's",82,12,"Indiana",53,1),(2022,4,"UCLA",57,13,"Akron",53,1),
    (2022,6,"Texas",81,11,"Virginia Tech",73,1),(2022,3,"Purdue",78,14,"Yale",56,1),
    (2022,7,"Murray St",92,10,"San Francisco",87,1),(2022,2,"Kentucky",79,15,"Saint Peter's",85,1),
    (2022,1,"Gonzaga",82,9,"Memphis",78,2),(2022,5,"Connecticut",52,4,"Arkansas",53,2),
    (2022,6,"Alabama",59,3,"Texas Tech",85,2),(2022,7,"Michigan St",68,2,"Duke",85,2),
    (2022,1,"Kansas",79,9,"Creighton",72,2),(2022,5,"Iowa",73,4,"Providence",54,2),
    (2022,6,"LSU",44,3,"Wisconsin",45,2),(2022,10,"Miami FL",79,2,"Auburn",61,2),
    (2022,1,"Arizona",85,9,"TCU",80,2),(2022,5,"Houston",68,4,"Illinois",53,2),
    (2022,11,"Michigan",76,3,"Tennessee",68,2),(2022,10,"Loyola Chicago",54,2,"Villanova",71,2),
    (2022,1,"Baylor",70,8,"North Carolina",93,2),(2022,5,"Saint Mary's",52,4,"UCLA",67,2),
    (2022,6,"Texas",56,3,"Purdue",81,2),(2022,15,"Saint Peter's",70,7,"Murray St",60,2),
    (2022,1,"Gonzaga",82,4,"Arkansas",74,3),(2022,3,"Texas Tech",55,2,"Duke",78,3),
    (2022,1,"Kansas",66,4,"Providence",61,3),(2022,10,"Miami FL",70,11,"Iowa St",56,3),
    (2022,1,"Arizona",72,5,"Houston",60,3),(2022,11,"Michigan",76,2,"Villanova",63,3),
    (2022,8,"North Carolina",73,4,"UCLA",66,3),(2022,3,"Purdue",67,15,"Saint Peter's",64,3),
    (2022,1,"Gonzaga",66,2,"Duke",81,4),(2022,1,"Kansas",76,10,"Miami FL",50,4),
    (2022,1,"Arizona",49,2,"Villanova",50,4),(2022,8,"North Carolina",69,15,"Saint Peter's",49,4),
    (2022,2,"Duke",52,8,"North Carolina",81,5),(2022,1,"Kansas",81,2,"Villanova",65,5),
    (2022,1,"Kansas",72,8,"North Carolina",69,6),
    # 2023 — UConn championship
    (2023,1,"Alabama",96,16,"Texas A&M-CC",75,1),(2023,8,"Maryland",67,9,"West Virginia",65,1),
    (2023,5,"San Diego St",63,12,"Charleston",57,1),(2023,4,"Virginia",51,13,"Furman",68,1),
    (2023,6,"Creighton",72,11,"NC State",63,1),(2023,3,"Baylor",74,14,"UC Santa Barbara",56,1),
    (2023,7,"Missouri",51,10,"Utah St",65,1),(2023,2,"Arizona",85,15,"Princeton",80,1),
    (2023,1,"Houston",63,16,"Northern Kentucky",52,1),(2023,8,"Iowa",83,9,"Auburn",75,1),
    (2023,5,"Miami FL",63,12,"Drake",56,1),(2023,4,"Indiana",63,13,"Kent St",71,1),
    (2023,6,"Iowa St",41,11,"Pittsburgh",59,1),(2023,3,"Xavier",72,14,"Kennesaw St",67,1),
    (2023,7,"Texas A&M",56,10,"Penn St",76,1),(2023,2,"Texas",81,15,"Colgate",61,1),
    (2023,1,"Purdue",63,16,"Fairleigh Dickinson",58,1),(2023,8,"Memphis",59,9,"FAU",66,1),
    (2023,5,"Duke",74,12,"Oral Roberts",51,1),(2023,4,"Tennessee",58,13,"Louisiana",55,1),
    (2023,6,"Kentucky",61,11,"Providence",53,1),(2023,3,"Kansas St",77,14,"Montana St",65,1),
    (2023,7,"Michigan St",72,10,"USC",62,1),(2023,2,"Marquette",78,15,"Vermont",61,1),
    (2023,1,"Kansas",96,16,"Howard",68,1),(2023,8,"Arkansas",73,9,"Illinois",63,1),
    (2023,5,"Saint Mary's",63,12,"VCU",51,1),(2023,4,"Connecticut",87,13,"Iona",63,1),
    (2023,6,"TCU",72,11,"Arizona St",70,1),(2023,3,"Gonzaga",82,14,"Grand Canyon",70,1),
    (2023,7,"Northwestern",75,10,"Boise St",67,1),(2023,2,"UCLA",86,15,"UNC Asheville",53,1),
    (2023,1,"Alabama",73,8,"Maryland",51,2),(2023,5,"San Diego St",75,13,"Furman",52,2),
    (2023,6,"Creighton",85,3,"Baylor",76,2),(2023,2,"Arizona",55,15,"Princeton",78,2),
    (2023,1,"Houston",56,8,"Iowa",57,2),(2023,5,"Miami FL",85,13,"Kent St",66,2),
    (2023,11,"Pittsburgh",59,3,"Xavier",84,2),(2023,2,"Texas",71,10,"Penn St",72,2),
    (2023,1,"Purdue",76,9,"FAU",78,2),(2023,5,"Duke",64,4,"Tennessee",65,2),
    (2023,6,"Kentucky",68,3,"Kansas St",75,2),(2023,2,"Marquette",67,7,"Michigan St",69,2),
    (2023,1,"Kansas",61,8,"Arkansas",72,2),(2023,5,"Saint Mary's",60,4,"Connecticut",70,2),
    (2023,6,"TCU",52,3,"Gonzaga",84,2),(2023,7,"Northwestern",65,2,"UCLA",68,2),
    (2023,1,"Alabama",73,5,"San Diego St",71,3),(2023,6,"Creighton",72,15,"Princeton",66,3),
    (2023,5,"Miami FL",58,5,"Houston",56,3),(2023,3,"Xavier",62,10,"Penn St",51,3),
    (2023,9,"FAU",62,4,"Tennessee",55,3),(2023,3,"Kansas St",67,7,"Michigan St",50,3),
    (2023,4,"Connecticut",82,8,"Arkansas",52,3),(2023,3,"Gonzaga",79,2,"UCLA",76,3),
    (2023,1,"Alabama",79,6,"Creighton",72,4),(2023,5,"Miami FL",72,3,"Xavier",56,4),
    (2023,9,"FAU",79,3,"Kansas St",76,4),(2023,4,"Connecticut",82,3,"Gonzaga",54,4),
    (2023,5,"San Diego St",72,9,"FAU",71,5),(2023,4,"Connecticut",72,5,"Miami FL",59,5),
    (2023,4,"Connecticut",76,5,"San Diego St",59,6),
    # 2024 — UConn repeat
    (2024,1,"Houston",62,16,"Longwood",52,1),(2024,8,"Nebraska",43,9,"Texas A&M",98,1),
    (2024,5,"Wisconsin",87,12,"James Madison",69,1),(2024,4,"Duke",64,13,"Vermont",47,1),
    (2024,6,"Texas Tech",56,11,"NC State",80,1),(2024,3,"Kentucky",85,14,"Oakland",89,1),
    (2024,7,"Florida",83,10,"Colorado",86,1),(2024,2,"Marquette",87,15,"Western Kentucky",69,1),
    (2024,1,"UConn",91,16,"Stetson",52,1),(2024,8,"FAU",67,9,"Northwestern",68,1),
    (2024,5,"San Diego St",69,12,"UAB",65,1),(2024,4,"Auburn",76,13,"Yale",78,1),
    (2024,6,"BYU",61,11,"Duquesne",71,1),(2024,3,"Illinois",85,14,"Morehead St",69,1),
    (2024,7,"Washington St",55,10,"Drake",53,1),(2024,2,"Iowa St",79,15,"South Dakota St",56,1),
    (2024,1,"North Carolina",90,16,"Wagner",62,1),(2024,8,"Mississippi St",63,9,"Michigan St",69,1),
    (2024,5,"Saint Mary's",63,12,"Grand Canyon",56,1),(2024,4,"Alabama",109,13,"Charleston",96,1),
    (2024,6,"Clemson",77,11,"New Mexico",56,1),(2024,3,"Baylor",56,14,"Colgate",67,1),
    (2024,7,"Dayton",56,10,"Nevada",63,1),(2024,2,"Arizona",78,15,"Long Beach St",57,1),
    (2024,1,"Purdue",78,16,"Grambling",50,1),(2024,8,"Utah St",55,9,"TCU",69,1),
    (2024,5,"Gonzaga",86,12,"McNeese",65,1),(2024,4,"Kansas",93,13,"Samford",89,1),
    (2024,6,"South Carolina",65,11,"Oregon",87,1),(2024,3,"Creighton",85,14,"Akron",45,1),
    (2024,7,"Texas",56,10,"Colorado St",67,1),(2024,2,"Tennessee",92,15,"Saint Peter's",62,1),
    (2024,1,"Houston",65,9,"Texas A&M",64,2),(2024,5,"Wisconsin",41,4,"Duke",64,2),
    (2024,11,"NC State",74,14,"Oakland",53,2),(2024,10,"Colorado",57,2,"Marquette",81,2),
    (2024,1,"UConn",75,9,"Northwestern",58,2),(2024,5,"San Diego St",60,13,"Yale",57,2),
    (2024,11,"Duquesne",58,3,"Illinois",89,2),(2024,7,"Washington St",42,2,"Iowa St",67,2),
    (2024,1,"North Carolina",85,9,"Michigan St",69,2),(2024,5,"Saint Mary's",55,4,"Alabama",89,2),
    (2024,6,"Clemson",72,3,"Baylor",56,2),(2024,10,"Nevada",57,2,"Arizona",85,2),
    (2024,1,"Purdue",78,9,"TCU",50,2),(2024,5,"Gonzaga",82,4,"Kansas",84,2),
    (2024,11,"Oregon",52,3,"Creighton",86,2),(2024,10,"Colorado St",56,2,"Tennessee",75,2),
    (2024,1,"Houston",65,4,"Duke",51,3),(2024,11,"NC State",76,2,"Marquette",64,3),
    (2024,1,"UConn",82,5,"San Diego St",52,3),(2024,3,"Illinois",72,2,"Iowa St",82,3),
    (2024,1,"North Carolina",72,4,"Alabama",89,3),(2024,6,"Clemson",60,2,"Arizona",77,3),
    (2024,1,"Purdue",80,4,"Kansas",68,3),(2024,3,"Creighton",55,2,"Tennessee",76,3),
    (2024,1,"Houston",54,11,"NC State",65,4),(2024,1,"UConn",77,2,"Iowa St",52,4),
    (2024,4,"Alabama",89,2,"Arizona",72,4),(2024,1,"Purdue",72,2,"Tennessee",66,4),
    (2024,11,"NC State",63,1,"Purdue",67,5),(2024,1,"UConn",86,4,"Alabama",72,5),
    (2024,1,"UConn",75,1,"Purdue",60,6),
    # 2025 — First round only (tournament in progress)
    (2025,1,"Auburn",74,16,"Alabama St",46,1),(2025,8,"Louisville",82,9,"Creighton",67,1),
    (2025,5,"Michigan",78,12,"UC San Diego",53,1),(2025,4,"Texas A&M",56,13,"Yale",68,1),
    (2025,6,"Ole Miss",59,11,"North Carolina",63,1),(2025,3,"Iowa St",77,14,"Lipscomb",56,1),
    (2025,7,"St. John's",80,10,"Vanderbilt",57,1),(2025,2,"Michigan St",75,15,"Bryant",48,1),
    (2025,1,"Duke",89,16,"American",42,1),(2025,8,"Mississippi St",58,9,"Baylor",72,1),
    (2025,5,"Oregon",72,12,"Liberty",64,1),(2025,4,"Arizona",73,13,"Akron",55,1),
    (2025,6,"Illinois",78,11,"Texas",63,1),(2025,3,"Wisconsin",67,14,"Montana",54,1),
    (2025,7,"UCLA",65,10,"UCSD",51,1),(2025,2,"Alabama",85,15,"Robert Morris",47,1),
    (2025,1,"Florida",81,16,"Norfolk St",54,1),(2025,8,"UConn",73,9,"Oklahoma",66,1),
    (2025,5,"Memphis",68,12,"Colorado St",61,1),(2025,4,"Maryland",69,13,"Grand Canyon",53,1),
    (2025,6,"Missouri",62,11,"Drake",71,1),(2025,3,"Texas Tech",82,14,"UNC Wilmington",55,1),
    (2025,7,"Kansas",68,10,"Arkansas",73,1),(2025,2,"St. John's",79,15,"Nebraska Omaha",52,1),
    (2025,1,"Houston",82,16,"SIU Edwardsville",48,1),(2025,8,"Gonzaga",69,9,"Georgia",65,1),
    (2025,5,"Clemson",73,12,"McNeese",67,1),(2025,4,"Purdue",77,13,"High Point",56,1),
    (2025,6,"BYU",64,11,"VCU",73,1),(2025,3,"Marquette",80,14,"New Mexico St",58,1),
    (2025,7,"Kansas St",61,10,"Butler",55,1),(2025,2,"Tennessee",88,15,"Wofford",53,1),
]

all_games = []
for g in RAW_GAMES:
    year, s1, t1, sc1, s2, t2, sc2, rnd = g
    all_games.append(build_game(year, s1, t1, sc1, s2, t2, sc2, rnd))

hist_df = pd.DataFrame(all_games)
print(f"Loaded {len(hist_df)} historical games across {hist_df['year'].nunique()} tournaments")
print(f"Years: {sorted(hist_df['year'].unique())}")

---
## LIVE ODDS — Run this to refresh

In [ ]:
#@title **Fetch Live Vegas Lines** (click to refresh)

SPORT = "basketball_ncaab"
BASE_URL = "https://api.the-odds-api.com/v4/sports"

def implied_probability(american_odds):
    if american_odds > 0:
        return 100 / (american_odds + 100)
    else:
        return abs(american_odds) / (abs(american_odds) + 100)

def fetch_live_odds(api_key):
    url = f"{BASE_URL}/{SPORT}/odds"
    params = {
        "apiKey": api_key, "regions": "us",
        "markets": "spreads,totals,h2h", "oddsFormat": "american",
    }
    resp = requests.get(url, params=params, timeout=30)
    if resp.status_code != 200:
        print(f"API error {resp.status_code}: {resp.text}")
        return []
    remaining = resp.headers.get('x-requests-remaining', '?')
    print(f"API requests remaining this month: {remaining}")
    return resp.json()

def parse_games(data):
    games = []
    for event in data:
        game = {
            'time': event['commence_time'][:16].replace('T',' '),
            'away': event['away_team'],
            'home': event['home_team'],
        }
        spreads, totals, mls = {}, {}, {}
        for book in event.get('bookmakers', []):
            bk = book['key']
            for mkt in book.get('markets', []):
                if mkt['key'] == 'spreads':
                    for o in mkt['outcomes']:
                        side = 'home' if o['name'] == event['home_team'] else 'away'
                        spreads.setdefault(side, []).append(o.get('point', 0))
                        spreads[f"{bk}_{side}"] = o.get('point', 0)
                elif mkt['key'] == 'totals':
                    for o in mkt['outcomes']:
                        if o['name'] == 'Over':
                            totals.setdefault('over', []).append(o.get('point', 0))
                            totals[f"{bk}"] = o.get('point', 0)
                elif mkt['key'] == 'h2h':
                    for o in mkt['outcomes']:
                        side = 'home' if o['name'] == event['home_team'] else 'away'
                        mls.setdefault(side, []).append(o.get('price', 0))
                        mls[f"{bk}_{side}"] = o.get('price', 0)

        # Consensus lines
        if 'away' in spreads:
            game['away_spread'] = round(np.mean(spreads['away']), 1)
            game['home_spread'] = round(np.mean(spreads.get('home', [0])), 1)
        if 'over' in totals:
            game['total'] = round(np.mean(totals['over']), 1)
        if 'home' in mls:
            game['home_ml'] = round(np.mean(mls['home']))
            game['away_ml'] = round(np.mean(mls.get('away', [0])))

        # Determine favorite
        if 'away_spread' in game:
            if game['away_spread'] < 0:
                game['favorite'] = game['away']
                game['underdog'] = game['home']
                game['spread'] = game['away_spread']
            else:
                game['favorite'] = game['home']
                game['underdog'] = game['away']
                game['spread'] = game['home_spread']

        # Book-by-book data for line shopping
        game['_spreads'] = spreads
        game['_totals'] = totals
        game['_mls'] = mls
        games.append(game)
    return games

live_games = []
if ODDS_API_KEY:
    raw_data = fetch_live_odds(ODDS_API_KEY)
    if raw_data:
        live_games = parse_games(raw_data)
        live_df = pd.DataFrame([{k:v for k,v in g.items() if not k.startswith('_')} for g in live_games])

        # Filter: NCAA Tournament games only
        tourney_mask = live_df.apply(lambda r: is_likely_tourney(r.get('away',''), r.get('home','')), axis=1)
        nit_count = (~tourney_mask).sum()
        live_df = live_df[tourney_mask]
        live_games = [g for g in live_games if is_likely_tourney(g.get('away',''), g.get('home',''))]
        if nit_count > 0:
            print(f'Filtered out {nit_count} non-tournament games (NIT/CBI/etc.)')
        print(f"\nFound {len(live_df)} games with odds\n")
        display(live_df[['time','away','home','spread','favorite','underdog','total','home_ml','away_ml']]
               .sort_values('time'))
    else:
        print("No games found. Tournament games may not be posted yet.")
else:
    print("No API key set — skipping live odds. Set key in Step 1 above.")

---
## VALUE BET FINDER
Compares live lines against 10 years of historical data

In [ ]:
#@title **Find Value Bets** (run after fetching live odds)

def find_value(live_games, hist_df, min_edge=0.03):
    """Compare live lines to historical trends."""
    value_bets = []
    
    for game in live_games:
        if 'spread' not in game or 'total' not in game:
            continue
        
        spread = abs(game['spread'])
        total_line = game['total']
        matchup_str = f"{game['away']} vs {game['home']}"
        
        # Find historical games with similar spread
        similar = hist_df[
            (hist_df['estimated_spread'] >= spread - 2.5) &
            (hist_df['estimated_spread'] <= spread + 2.5)
        ]
        if len(similar) < 10:
            continue
        
        # Dog cover rate
        dog_cover = 1 - similar['favorite_covered'].mean()
        if dog_cover > 0.524 + min_edge:
            value_bets.append({
                'Game': matchup_str,
                'Bet': f"{game['underdog']} +{spread:.1f}",
                'Type': 'Dog ATS',
                'Hist Cover%': f"{dog_cover:.0%}",
                'Edge': f"{dog_cover - 0.524:.1%}",
                'Sample': len(similar),
                'Rating': '***' if dog_cover > 0.60 else '**' if dog_cover > 0.55 else '*',
            })
        
        # Totals
        over_rate = (similar['total_points'] > total_line).mean()
        under_rate = 1 - over_rate
        
        if over_rate > 0.524 + min_edge:
            value_bets.append({
                'Game': matchup_str,
                'Bet': f"Over {total_line}",
                'Type': 'Over',
                'Hist Cover%': f"{over_rate:.0%}",
                'Edge': f"{over_rate - 0.524:.1%}",
                'Sample': len(similar),
                'Rating': '***' if over_rate > 0.60 else '**' if over_rate > 0.55 else '*',
            })
        if under_rate > 0.524 + min_edge:
            value_bets.append({
                'Game': matchup_str,
                'Bet': f"Under {total_line}",
                'Type': 'Under',
                'Hist Cover%': f"{under_rate:.0%}",
                'Edge': f"{under_rate - 0.524:.1%}",
                'Sample': len(similar),
                'Rating': '***' if under_rate > 0.60 else '**' if under_rate > 0.55 else '*',
            })
        
        # Upset ML check
        if spread >= 4:
            upset_rate = similar['upset'].mean()
            if 'away_ml' in game and 'home_ml' in game:
                dog_ml = game['away_ml'] if game['underdog'] == game['away'] else game['home_ml']
                if dog_ml > 0:
                    implied = implied_probability(dog_ml)
                    if upset_rate > implied + min_edge:
                        value_bets.append({
                            'Game': matchup_str,
                            'Bet': f"{game['underdog']} ML ({dog_ml:+d})",
                            'Type': 'Dog ML',
                            'Hist Cover%': f"{upset_rate:.0%}",
                            'Edge': f"{upset_rate - implied:.1%} vs implied {implied:.0%}",
                            'Sample': len(similar),
                            'Rating': '***' if (upset_rate - implied) > 0.10 else '**' if (upset_rate - implied) > 0.05 else '*',
                        })
    
    return value_bets

if live_games:
    value = find_value(live_games, hist_df)
    if value:
        value_df = pd.DataFrame(value)
        print(f"Found {len(value)} potential value bets\n")
        print("Rating: *** = strong, ** = moderate, * = slight edge\n")
        display(value_df.sort_values('Rating', ascending=False))
    else:
        print("No clear value bets found at current lines.")
        print("Try adjusting min_edge parameter in find_value() to 0.02 for looser criteria.")
else:
    print("No live games loaded. Fetch live odds first (cell above).")
    print("\nIn the meantime, here are general historical edges:")
    print(f"  - Dogs cover {1-hist_df['favorite_covered'].mean():.0%} ATS overall")
    print(f"  - 5v12 upset rate: {hist_df[hist_df['matchup']=='5v12']['upset'].mean():.0%}")
    print(f"  - 8v9 upset rate: {hist_df[hist_df['matchup']=='8v9']['upset'].mean():.0%}")

---
## LINE SHOPPING
Find the best number across sportsbooks

In [ ]:
#@title **Line Shopping — Best Numbers by Book**

if live_games:
    print("BEST AVAILABLE LINES BY GAME")
    print("=" * 70)
    
    for game in live_games:
        print(f"\n{game['away']} @ {game['home']}")
        print(f"  Consensus: {game.get('favorite','?')} {game.get('spread','?')} | Total {game.get('total','?')}")
        
        # Best spreads
        sp = game.get('_spreads', {})
        book_spreads_away = {k: v for k, v in sp.items() if k.endswith('_away') and isinstance(v, (int, float))}
        book_spreads_home = {k: v for k, v in sp.items() if k.endswith('_home') and isinstance(v, (int, float))}
        
        if book_spreads_away:
            best_away_book = max(book_spreads_away, key=book_spreads_away.get)
            best_home_book = max(book_spreads_home, key=book_spreads_home.get)
            print(f"  Best {game['away']} spread: {book_spreads_away[best_away_book]:+.1f} ({best_away_book.replace('_away','')})")
            print(f"  Best {game['home']} spread: {book_spreads_home[best_home_book]:+.1f} ({best_home_book.replace('_home','')})")
        
        # Best totals
        tot = game.get('_totals', {})
        book_totals = {k: v for k, v in tot.items() if k not in ('over',) and isinstance(v, (int, float))}
        if book_totals:
            best_over_book = max(book_totals, key=book_totals.get)
            best_under_book = min(book_totals, key=book_totals.get)
            print(f"  Best Over: {book_totals[best_over_book]} ({best_over_book})")
            print(f"  Best Under: {book_totals[best_under_book]} ({best_under_book})")
        
        # Best MLs
        ml = game.get('_mls', {})
        book_ml_home = {k: v for k, v in ml.items() if k.endswith('_home') and isinstance(v, (int, float))}
        book_ml_away = {k: v for k, v in ml.items() if k.endswith('_away') and isinstance(v, (int, float))}
        if book_ml_home:
            best_home_ml = max(book_ml_home, key=book_ml_home.get)
            best_away_ml = max(book_ml_away, key=book_ml_away.get)
            print(f"  Best {game['home']} ML: {book_ml_home[best_home_ml]:+d} ({best_home_ml.replace('_home','')})")
            print(f"  Best {game['away']} ML: {book_ml_away[best_away_ml]:+d} ({best_away_ml.replace('_away','')})")
else:
    print("Fetch live odds first.")

---
## FIRST HALF (1H) ANALYSIS
Live 1H spreads & totals + historical tendencies

In [ ]:
#@title **Fetch Live 1H Lines** (uses ~1 API request per game)
#@markdown Note: This fetches 1H lines per-game. Uses more API quota than full-game lines.
FETCH_1H = True  #@param {type:"boolean"}
MAX_GAMES = 20  #@param {type:"integer"}

FIRST_HALF_TOTAL_RATIO = 0.48  # 1H is ~48% of full game total
FIRST_HALF_SPREAD_RATIO = 0.52  # 1H spread is ~52% of full game spread

h1_games = []
if FETCH_1H and ODDS_API_KEY:
    # Get event IDs
    events_resp = requests.get(
        f'{BASE_URL}/{SPORT}/events',
        params={'apiKey': ODDS_API_KEY}, timeout=30
    )
    events = events_resp.json() if events_resp.status_code == 200 else []
    remaining = int(events_resp.headers.get('x-requests-remaining', 500))
    events = events[:min(MAX_GAMES, max(0, remaining - 5))]
    print(f'Fetching 1H lines for {len(events)} games (API remaining: {remaining})...')

    for i, event in enumerate(events):
        resp = requests.get(
            f"{BASE_URL}/{SPORT}/events/{event['id']}/odds",
            params={'apiKey': ODDS_API_KEY, 'regions': 'us',
                    'markets': 'h2h_h1,spreads_h1,totals_h1', 'oddsFormat': 'american'},
            timeout=30
        )
        if resp.status_code != 200: continue
        data = resp.json()
        game = {'away': data.get('away_team',''), 'home': data.get('home_team',''),
                'time': data.get('commence_time','')[:16].replace('T',' ')}
        h1_sp_a, h1_sp_h, h1_tot, h1_ml_h, h1_ml_a = [], [], [], [], []
        for book in data.get('bookmakers', []):
            for mkt in book.get('markets', []):
                if mkt['key'] == 'spreads_h1':
                    for o in mkt['outcomes']:
                        (h1_sp_h if o['name']==data.get('home_team','') else h1_sp_a).append(o.get('point',0))
                elif mkt['key'] == 'totals_h1':
                    for o in mkt['outcomes']:
                        if o['name']=='Over': h1_tot.append(o.get('point',0))
                elif mkt['key'] == 'h2h_h1':
                    for o in mkt['outcomes']:
                        (h1_ml_h if o['name']==data.get('home_team','') else h1_ml_a).append(o.get('price',0))
        if h1_sp_a: game['h1_away_spread'] = round(np.mean(h1_sp_a), 1)
        if h1_sp_h: game['h1_home_spread'] = round(np.mean(h1_sp_h), 1)
        if h1_tot: game['h1_total'] = round(np.mean(h1_tot), 1)
        if h1_ml_h: game['h1_home_ml'] = round(np.mean(h1_ml_h))
        if h1_ml_a: game['h1_away_ml'] = round(np.mean(h1_ml_a))
        if h1_sp_a and h1_sp_h:
            if np.mean(h1_sp_a) < 0:
                game['h1_favorite'] = game['away']; game['h1_underdog'] = game['home']; game['h1_spread'] = game['h1_away_spread']
            else:
                game['h1_favorite'] = game['home']; game['h1_underdog'] = game['away']; game['h1_spread'] = game['h1_home_spread']
        h1_games.append(game)

    h1_df = pd.DataFrame(h1_games)
    remaining = resp.headers.get('x-requests-remaining', '?')
    print(f'Fetched 1H lines for {len(h1_df)} games. API remaining: {remaining}\n')
    cols = [c for c in ['time','away','home','h1_spread','h1_favorite','h1_total','h1_home_ml','h1_away_ml'] if c in h1_df.columns]
    display(h1_df[cols].sort_values('time'))
else:
    print('Set FETCH_1H=True and provide API key to fetch 1H lines.')

In [ ]:
#@title **1H Value Bet Finder**

def find_1h_value(h1_games, live_games, hist_df):
    """Compare live 1H lines to model estimates derived from full-game data."""
    value = []
    # Build lookup from full-game lines
    fg_lookup = {}
    for g in live_games:
        key = (g.get('away',''), g.get('home',''))
        fg_lookup[key] = g

    for game in h1_games:
        key = (game.get('away',''), game.get('home',''))
        fg = fg_lookup.get(key, {})
        fg_spread = abs(fg.get('spread', 0))
        fg_total = fg.get('total', 0)
        if not fg_total: continue
        matchup_str = f"{game['away']} vs {game['home']}"

        # Model estimates
        est_1h_total = fg_total * FIRST_HALF_TOTAL_RATIO
        est_1h_spread = fg_spread * FIRST_HALF_SPREAD_RATIO

        # Compare to live 1H total
        h1_total = game.get('h1_total', 0)
        if h1_total:
            # Historical: what % of similar games had est 1H total above this line?
            similar = hist_df[(hist_df['total_points'] >= fg_total-10) & (hist_df['total_points'] <= fg_total+10)]
            if len(similar) >= 10:
                est_1h_totals = similar['total_points'] * FIRST_HALF_TOTAL_RATIO
                over_rate = (est_1h_totals > h1_total).mean()
                under_rate = 1 - over_rate
                if over_rate > 0.57:
                    value.append({'Game': matchup_str, 'Bet': f'1H Over {h1_total}', 'Type': '1H Over',
                        'Edge': f'{over_rate:.0%} hist rate (model: {est_1h_total:.1f})',
                        'Rating': '***' if over_rate > 0.65 else '**' if over_rate > 0.60 else '*'})
                if under_rate > 0.57:
                    value.append({'Game': matchup_str, 'Bet': f'1H Under {h1_total}', 'Type': '1H Under',
                        'Edge': f'{under_rate:.0%} hist rate (model: {est_1h_total:.1f})',
                        'Rating': '***' if under_rate > 0.65 else '**' if under_rate > 0.60 else '*'})

        # Compare to live 1H spread
        h1_spread = abs(game.get('h1_spread', 0))
        if h1_spread and est_1h_spread:
            diff = h1_spread - est_1h_spread
            if diff > 1.5:  # Live spread is wider than model — dog value
                value.append({'Game': matchup_str,
                    'Bet': f"1H {game.get('h1_underdog','Dog')} +{h1_spread}",
                    'Type': '1H Dog ATS',
                    'Edge': f'Getting {diff:.1f} extra pts (model: {est_1h_spread:.1f})',
                    'Rating': '**' if diff > 2.5 else '*'})
            elif diff < -1.5:  # Live spread tighter than model — fav value
                value.append({'Game': matchup_str,
                    'Bet': f"1H {game.get('h1_favorite','Fav')} {game.get('h1_spread', 0)}",
                    'Type': '1H Fav ATS',
                    'Edge': f'Laying {abs(diff):.1f} fewer pts (model: -{est_1h_spread:.1f})',
                    'Rating': '**' if abs(diff) > 2.5 else '*'})
    return value

if h1_games and live_games:
    h1_value = find_1h_value(h1_games, live_games, hist_df)
    if h1_value:
        h1_vdf = pd.DataFrame(h1_value)
        print(f'Found {len(h1_value)} potential 1H value bets\n')
        display(h1_vdf.sort_values('Rating', ascending=False))
    else:
        print('No 1H value bets found at current lines.')
else:
    print('Fetch 1H lines first (cell above).')

In [ ]:
#@title **1H Historical Tendencies**

print('FIRST HALF TOURNAMENT TENDENCIES (from full-game data)')
print('=' * 60)

hist_df['est_1h_total'] = hist_df['total_points'] * FIRST_HALF_TOTAL_RATIO
hist_df['est_1h_margin'] = hist_df['actual_margin'] * FIRST_HALF_SPREAD_RATIO

r1 = hist_df[hist_df['round_num'] == 1]
late = hist_df[hist_df['round_num'] >= 3]

print(f'\n--- ESTIMATED 1H TOTALS ---')
print(f'Overall avg 1H total: {hist_df["est_1h_total"].mean():.1f}')
print(f'First Round avg 1H total: {r1["est_1h_total"].mean():.1f}')
print(f'Sweet 16+ avg 1H total: {late["est_1h_total"].mean():.1f}')

print(f'\n--- 1H SPREAD TENDENCIES ---')
print(f'First Round: favorite leads at half {(r1["est_1h_margin"] > 0).mean():.0%} of games')
print(f'Sweet 16+: favorite leads at half {(late["est_1h_margin"] > 0).mean():.0%} of games')

# 1H tendencies by matchup
print(f'\n--- 1H DOG VALUE BY MATCHUP ---')
r1_1h = r1.groupby('matchup').agg(
    n=('est_1h_margin', 'count'),
    dog_leading_1h=('est_1h_margin', lambda x: (x < 0).mean()),
    avg_1h_margin=('est_1h_margin', 'mean'),
).reset_index()
r1_1h = r1_1h[r1_1h['n'] >= 5].sort_values('dog_leading_1h', ascending=False)
for _, row in r1_1h.head(8).iterrows():
    print(f"  {row['matchup']}: dog leads at half {row['dog_leading_1h']:.0%} "
          f"(avg margin: {row['avg_1h_margin']:+.1f}, n={int(row['n'])})")

print(f'\n--- KEY 1H INSIGHTS ---')
print(f'1. 1H unders hit more often in March Madness (nerves slow early scoring)')
print(f'2. Underdogs keep it closer in the 1H than the full game')
print(f'3. Big favorites (10+ pts FG) often dont separate until the 2H')
print(f'4. Sweet 16+ games have tighter 1H scoring (better defense, prep time)')
print(f'5. 1H spread ~ 52% of full-game spread, 1H total ~ 48% of full-game total')

---
## BET SHEET — Today's Best Plays
Unified graded output: full game + first half, sorted by strength

In [ ]:
#@title **Generate Bet Sheet** { display-mode: "form" }
#@markdown Adjust the minimum grade to filter bets:
MIN_GRADE = "B" #@param ["A+", "A", "A-", "B+", "B", "B-", "C+", "C"]

GRADE_ORDER = ['A+','A','A-','B+','B','B-','C+','C','C-','D+','D','F']
GRADE_CUTOFF = GRADE_ORDER.index(MIN_GRADE)

def score_bet(edge_pct, sample_size, bet_type):
    """
    Score a bet 0-100 based on:
      - Edge strength (how far above break-even)
      - Sample size (more data = more reliable)
      - Bet type bonus (some markets are sharper)
    """
    # Edge component (0-50 pts): break-even is 52.4%, so edge = rate - 0.524
    edge_score = min(50, max(0, (edge_pct - 0.524) * 200))

    # Sample component (0-30 pts)
    if sample_size >= 100: sample_score = 30
    elif sample_size >= 50: sample_score = 25
    elif sample_size >= 20: sample_score = 20
    elif sample_size >= 10: sample_score = 10
    else: sample_score = 5

    # Type bonus (0-20 pts): some bet types are historically more reliable
    type_bonus = {'Dog ATS': 15, 'Under': 12, 'Over': 10, 'Dog ML': 8,
                  '1H Under': 18, '1H Over': 14, '1H Dog ATS': 12, '1H Fav ATS': 10}.get(bet_type, 10)

    return edge_score + sample_score + type_bonus

def score_to_grade(score):
    if score >= 85: return 'A+'
    elif score >= 78: return 'A'
    elif score >= 72: return 'A-'
    elif score >= 65: return 'B+'
    elif score >= 55: return 'B'
    elif score >= 48: return 'B-'
    elif score >= 40: return 'C+'
    elif score >= 32: return 'C'
    elif score >= 25: return 'C-'
    else: return 'D'

def parse_edge_pct(edge_str):
    """Extract a numeric rate from various edge string formats."""
    import re
    m = re.search(r'(\d+)%', str(edge_str))
    return int(m.group(1)) / 100 if m else 0.5

# ---- Collect all bets ----
all_bets = []

# Full-game value bets
if live_games:
    fg_value = find_value(live_games, hist_df, min_edge=0.02)
    for bet in fg_value:
        rate = parse_edge_pct(bet.get('Hist Cover%', '50%'))
        sample = bet.get('Sample', 10)
        btype = bet.get('Type', 'Unknown')
        score = score_bet(rate, sample, btype)
        grade = score_to_grade(score)
        all_bets.append({
            'Grade': grade,
            'Score': score,
            'Game': bet['Game'],
            'Bet': bet['Bet'],
            'Type': btype,
            'Edge': bet.get('Hist Cover%', ''),
            'Detail': bet.get('Edge', ''),
            'N': sample,
        })

# 1H value bets
if h1_games and live_games:
    h1_val = find_1h_value(h1_games, live_games, hist_df)
    for bet in h1_val:
        rate = parse_edge_pct(bet.get('Edge', '50%'))
        score = score_bet(rate, 50, bet.get('Type', '1H'))
        grade = score_to_grade(score)
        all_bets.append({
            'Grade': grade,
            'Score': score,
            'Game': bet['Game'],
            'Bet': bet['Bet'],
            'Type': bet.get('Type', '1H'),
            'Edge': bet.get('Edge', ''),
            'Detail': '',
            'N': '-',
        })

# ---- Filter & Display ----
from datetime import datetime, timedelta
import re as _re

if all_bets:
    # Build time lookup from live games
    time_lookup = {}
    if live_games:
        for g in live_games:
            key = f"{g.get('away','')} vs {g.get('home','')}"
            time_lookup[key] = g.get('time', '')

    bets_df = pd.DataFrame(all_bets)
    bets_df = bets_df.sort_values('Score', ascending=False)

    # Add game time for each bet
    bets_df['game_time'] = bets_df['Game'].map(time_lookup).fillna('')

    # Add game time for day grouping
    if live_games:
    else:

    # Parse day from time string (format: '2026-03-19 11:15' or 'Thu 03/19 11:15AM')
    def parse_day(t):
        try:
            s = str(t).strip()
            if len(s) >= 10 and '-' in s:  # ISO: 2026-03-19 00:00
                from datetime import datetime as _dt, timedelta, timezone
                dt_utc = _dt.strptime(s[:16], '%Y-%m-%d %H:%M')
                dt_local = dt_utc - timedelta(hours=5)  # EST
                return dt_local.strftime('%A %m/%d')
            m = _re.search(r'(\d{2}/\d{2})', s)
            if m:
                dt = _dt.strptime(f'2026/{m.group(1)}', '%Y/%m/%d')
                return dt.strftime('%A %m/%d')
        except: pass
        return 'Unknown'

    # Extract clean time string for display
    def format_game_time(t):
        try:
            s = str(t).strip()
            if len(s) >= 10 and '-' in s:  # ISO: 2026-03-19 00:00
                from datetime import datetime as _dt, timedelta
                dt_utc = _dt.strptime(s[:16], '%Y-%m-%d %H:%M')
                dt_local = dt_utc - timedelta(hours=5)  # EST
                return dt_local.strftime('%I:%M %p').lstrip('0')
            m = _re.search(r'(\d{1,2}:\d{2}\s*[APap][Mm])', s)
            if m: return m.group(1)
        except: pass
        return ''

    bets_df['Time'] = bets_df['game_time'].apply(format_game_time)

    bets_df['day'] = bets_df['game_time'].apply(parse_day)

    # Filter by grade
    bets_df['grade_idx'] = bets_df['Grade'].apply(lambda g: GRADE_ORDER.index(g) if g in GRADE_ORDER else 99)
    filtered = bets_df[bets_df['grade_idx'] <= GRADE_CUTOFF].copy()

    if len(filtered) == 0:
        print(f'No bets at grade {MIN_GRADE} or above. Try lowering the threshold.')
    else:
        # Summary header
        grade_counts = filtered['Grade'].value_counts().reindex(GRADE_ORDER).dropna().astype(int)
        print(f'BET SHEET  |  {len(filtered)} plays at {MIN_GRADE} or better')
        print(f'Grades: {"  ".join(f"{g}: {c}" for g, c in grade_counts.items())}')
        print('=' * 80)

        # Group by day, then by grade within each day
        days_in_order = sorted(filtered['day'].unique(), 
                               key=lambda d: datetime.strptime(d.split()[-1], '%m/%d') if d != 'Unknown' else datetime.max)

        for day in days_in_order:
            day_bets = filtered[filtered['day'] == day]
            n_games = day_bets['Game'].nunique()
            print(f'\n{"#" * 70}')
            print(f'  {day.upper()}  ({n_games} games, {len(day_bets)} plays)')
            print(f'{"#" * 70}')

            for grade in GRADE_ORDER[:GRADE_CUTOFF+1]:
                grade_bets = day_bets[day_bets['Grade'] == grade]
                if len(grade_bets) == 0:
                    continue
                print(f'\n  --- {grade} ({len(grade_bets)} plays) ---')
                for _, row in grade_bets.iterrows():
                    time_short = ''
                    tm = _re.search(r'(\d{1,2}:\d{2}\s*[APap][Mm])', str(row.get('game_time','')))
                    if tm: time_short = tm.group(1)
                    print(f'    {row["Bet"]:<42} {row["Edge"]:<8} {time_short}')

        # Clean table at the end
        print(f'\n\n{"=" * 80}')
        print('FULL TABLE')
        print(f'{"=" * 80}')
        display(filtered[['Grade','day','Time','Bet','Type','Edge','Game','N']]
                .rename(columns={'day': 'Day'})
                .drop(columns=['Score','grade_idx','game_time'], errors='ignore'))
else:
    print('No bets to display. Run the live odds and 1H cells above first.')
    print('If you\'re offline, the Historical Cheat Sheet below still works.')


In [ ]:
#@title **Export Bet Sheet to Google Sheets** { display-mode: "form" }
#@markdown Creates a new Google Sheet with today's graded bets.
#@markdown You'll be prompted to authorize Google Drive access.
SHEET_NAME = "March Madness Bets"  #@param {type:"string"}

try:
    from google.colab import auth
    import gspread
    from google.auth import default
except ImportError:
    print('This cell only works in Google Colab.')
    print('To export locally, the CSV is saved below instead.')
    if 'filtered' in dir() and len(filtered) > 0:
        csv_path = 'bet_sheet.csv'
        filtered[['Grade','Bet','Type','Edge','Game','N']].to_csv(csv_path, index=False)
        print(f'Saved to {csv_path}')
    raise SystemExit

if 'filtered' not in dir() or len(filtered) == 0:
    print('No bets to export. Run the Bet Sheet cell above first.')
else:
    # Authenticate
    auth.authenticate_user()
    creds, _ = default()
    gc = gspread.authorize(creds)

    # Create or open sheet
    try:
        sh = gc.open(SHEET_NAME)
        ws = sh.sheet1
        ws.clear()
        print(f'Updating existing sheet: {SHEET_NAME}')
    except gspread.SpreadsheetNotFound:
        sh = gc.create(SHEET_NAME)
        ws = sh.sheet1
        # Make it accessible to anyone with the link
        sh.share('', perm_type='anyone', role='reader')
        print(f'Created new sheet: {SHEET_NAME}')

    # Prepare data
    export_cols = [c for c in ['Grade','day','Time','Bet','Type','Edge','Game','N'] if c in filtered.columns]
    export_df = filtered[export_cols].copy()
    export_df = export_df.reset_index(drop=True)

    # Write header row with formatting info
    from datetime import datetime
    header_rows = [
        [f'MARCH MADNESS BET SHEET — {datetime.now().strftime("%B %d, %Y %I:%M %p")}'],
        [f'Minimum grade: {MIN_GRADE}  |  Total plays: {len(export_df)}'],
        [''],
        ['Grade', 'Day', 'Time', 'Bet', 'Type', 'Hist Rate', 'Game', 'Sample'],
    ]

    # Data rows
    data_rows = export_df.values.tolist()

    # Write everything
    all_rows = header_rows + data_rows
    ws.update(f'A1:H{len(all_rows)}', all_rows)

    # Format header
    ws.format('A1:H1', {'textFormat': {'bold': True, 'fontSize': 14}})
    ws.format('A4:H4', {'textFormat': {'bold': True},
                         'backgroundColor': {'red': 0.2, 'green': 0.2, 'blue': 0.2},
                         'textFormat': {'bold': True, 'foregroundColor': {'red': 1, 'green': 1, 'blue': 1}}})

    # Color-code grades
    grade_colors = {
        'A+': {'red': 0.0, 'green': 0.8, 'blue': 0.0},
        'A':  {'red': 0.2, 'green': 0.7, 'blue': 0.1},
        'A-': {'red': 0.4, 'green': 0.6, 'blue': 0.1},
        'B+': {'red': 0.6, 'green': 0.6, 'blue': 0.0},
        'B':  {'red': 0.8, 'green': 0.5, 'blue': 0.0},
        'B-': {'red': 0.9, 'green': 0.4, 'blue': 0.0},
    }
    for row_idx, row_data in enumerate(data_rows):
        grade = row_data[0]
        if grade in grade_colors:
            cell_row = row_idx + 5  # offset for header rows
            ws.format(f'A{cell_row}', {
                'backgroundColor': grade_colors[grade],
                'textFormat': {'bold': True, 'foregroundColor': {'red': 1, 'green': 1, 'blue': 1}}
            })

    # Auto-resize columns
    ws.columns_auto_resize(0, 8)

    url = sh.url
    print(f'\nBet sheet exported to Google Sheets!')
    print(f'URL: {url}')
    print(f'\n{len(export_df)} bets written. Sheet is shared as view-only link.')
    print('Bookmark this URL to check on your phone at the casino.')

---
## HISTORICAL CHEAT SHEET
Quick reference for the casino floor

In [ ]:
#@title **Quick Reference Cheat Sheet**

print("MARCH MADNESS BETTING CHEAT SHEET (2016-2025)")
print("=" * 60)

# Spread insights
print(f"\n--- SPREAD (ATS) ---")
print(f"Overall favorite cover rate: {hist_df['favorite_covered'].mean():.0%}")
print(f"Overall underdog cover rate: {1-hist_df['favorite_covered'].mean():.0%}")

r1 = hist_df[hist_df['round_num'] == 1]
matchups = r1.groupby('matchup').agg(
    n=('favorite_covered','count'),
    dog_cover=('favorite_covered', lambda x: 1-x.mean())
).reset_index()
matchups = matchups[matchups['n'] >= 5].sort_values('dog_cover', ascending=False)

print(f"\nBest underdog ATS spots (First Round):")
for _, row in matchups.head(6).iterrows():
    print(f"  {row['matchup']}: dog covers {row['dog_cover']:.0%} (n={int(row['n'])})")

# Totals insights
print(f"\n--- TOTALS ---")
for rnd, name in [(1,'First Round'), (2,'Second Round'), (3,'Sweet 16'), (4,'Elite 8')]:
    rd = hist_df[hist_df['round_num'] == rnd]
    print(f"  {name}: avg={rd['total_points'].mean():.0f}, "
          f"O150={( rd['total_points']>150).mean():.0%}, "
          f"U130={(rd['total_points']<130).mean():.0%}")

# Upset rates
print(f"\n--- MONEYLINE UPSETS (First Round) ---")
r1_upsets = r1.groupby('matchup').agg(
    n=('upset','count'), upsets=('upset','sum')
).reset_index()
r1_upsets['rate'] = r1_upsets['upsets'] / r1_upsets['n']
r1_upsets = r1_upsets[r1_upsets['n'] >= 5].sort_values('rate', ascending=False)
for _, row in r1_upsets.iterrows():
    print(f"  {row['matchup']}: {row['rate']:.0%} ({int(row['upsets'])}/{int(row['n'])})")

print(f"\n--- RULES OF THUMB ---")
print(f"1. Underdogs cover ~59% ATS — lean dog in close matchups")
print(f"2. 5v12 and 6v11 are prime upset spots")
print(f"3. Scoring drops ~10pts from R1 to Elite 8 — lean under late")
print(f"4. Big favorites (15+ pt spreads) rarely cover")
print(f"5. 8v9 games are coin flips — take the dog if getting points")

In [ ]:
# Quick visualization for reference
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Dog cover rate by round
rounds = hist_df.groupby('round_name')['favorite_covered'].mean()
rounds = rounds.reindex(['First Round','Second Round','Sweet 16','Elite 8','Final Four','Championship'])
dog_rounds = 1 - rounds
colors = ['#2ecc71' if x > 0.524 else '#e74c3c' for x in dog_rounds]
axes[0].bar(range(len(dog_rounds)), dog_rounds, color=colors)
axes[0].set_xticks(range(len(dog_rounds)))
axes[0].set_xticklabels(['R1','R2','S16','E8','F4','Champ'], fontsize=10)
axes[0].axhline(y=0.524, color='orange', linestyle='--', label='Break-even')
axes[0].set_title('Dog Cover Rate by Round')
axes[0].set_ylabel('Cover %')
axes[0].legend()

# 2. Totals distribution
axes[1].hist(hist_df['total_points'], bins=25, color='#3498db', alpha=0.7, edgecolor='black')
axes[1].axvline(hist_df['total_points'].mean(), color='red', linestyle='--', 
                label=f"Mean: {hist_df['total_points'].mean():.0f}")
axes[1].set_title('Total Points Distribution')
axes[1].set_xlabel('Total Points')
axes[1].legend()

# 3. First round upset rates
r1_up = r1.groupby('matchup')['upset'].mean().sort_values(ascending=True)
r1_up = r1_up[r1.groupby('matchup')['upset'].count() >= 5]
axes[2].barh(r1_up.index, r1_up.values, color=plt.cm.RdYlGn(r1_up.values / r1_up.max()))
axes[2].set_title('First Round Upset Rate')
axes[2].set_xlabel('Upset %')

plt.suptitle('March Madness Quick Reference', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()